## First

In [ ]:
# ===================== CELL 1: build HOST-focused scene + store what will be shown to LLM =====================
import json, time
from datetime import datetime
import numpy as np
import pandas as pd

HOST_CSV = "host_stats_onos.csv"          # <-- your host stats CSV
HOST_META = "host_metadata.json"    # <-- optional (mac->host mapping)
WINDOW_MINUTES = 5
TREND_POINTS = 10
SCENE_LOG = "llm_scene_log.jsonl"
DECISION_LOG = "llm_decision_log.jsonl"

def now_iso(): 
    return datetime.now().isoformat()

def build_host_stats(df, trend_points=10):
    """
    Input: df already filtered to the desired time window
    Output: list of host objects (one per MAC)
    """
    hosts = []
    for mac, g in df.groupby("host_mac"):
        g = g.sort_values("timestamp").tail(trend_points)

        tx_pps = g["tx_pps"].astype(float).values
        rx_pps = g["rx_pps"].astype(float).values

        tx_kbps = g["tx_mbps"].astype(float).values * 1000.0
        rx_kbps = g["rx_mbps"].astype(float).values * 1000.0

        if len(tx_pps) < 2:
            continue

        host_obj = {
            "mac": str(mac),

            "tx_pps_trend": [round(x, 2) for x in tx_pps.tolist()],
            "rx_pps_trend": [round(x, 2) for x in rx_pps.tolist()],

            "tx_kbps_trend": [round(x, 2) for x in tx_kbps.tolist()],
            "rx_kbps_trend": [round(x, 2) for x in rx_kbps.tolist()],

            "tx_pps_mean": round(float(np.mean(tx_pps)), 2),
            "rx_pps_mean": round(float(np.mean(rx_pps)), 2),

            "tx_pps_std": round(float(np.std(tx_pps)), 3),
            "rx_pps_std": round(float(np.std(rx_pps)), 3),

            "tx_pps_max": round(float(np.max(tx_pps)), 2),
            "rx_pps_max": round(float(np.max(rx_pps)), 2),

            "tx_kbps_mean": round(float(np.mean(tx_kbps)), 2),
            "rx_kbps_mean": round(float(np.mean(rx_kbps)), 2),

            "tx_kbps_std": round(float(np.std(tx_kbps)), 3),
            "rx_kbps_std": round(float(np.std(rx_kbps)), 3),

            "tx_kbps_max": round(float(np.max(tx_kbps)), 2),
            "rx_kbps_max": round(float(np.max(rx_kbps)), 2),

            "tx_pps_delta": round(float(tx_pps[-1] - tx_pps[-2]), 2),
            "rx_pps_delta": round(float(rx_pps[-1] - rx_pps[-2]), 2),

            "tx_kbps_delta": round(float(tx_kbps[-1] - tx_kbps[-2]), 2),
            "rx_kbps_delta": round(float(rx_kbps[-1] - rx_kbps[-2]), 2),
        }

        hosts.append(host_obj)

    return hosts

def build_scene_and_prompt(model_name="llama3", last_k_decisions=10):
    # ---- read host stats ----
    df = pd.read_csv(HOST_CSV)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    if df.empty:
        return None, None

    # ---- filter last WINDOW_MINUTES ----
    t_end = df["timestamp"].max()
    t_min = t_end - pd.Timedelta(minutes=WINDOW_MINUTES)
    dfw = df[df["timestamp"] >= t_min].copy()
    if dfw.empty:
        return None, None
    
    # ---- metadata ----
    try:
        with open(HOST_META, "r", encoding="utf-8") as f:
            host_meta_obj = json.load(f)
    except FileNotFoundError:
        host_meta_obj = {"note": f"{HOST_META} not found"}

    # ---- build host stats via separate function ----
    hosts = build_host_stats(dfw, trend_points=TREND_POINTS)

    #///// filter //////
    hosts = [
        {
            "mac": h["mac"],
            "tx_pps_trend": h["tx_pps_trend"],
            "rx_pps_trend": h["rx_pps_trend"],
            "tx_kbps_trend": h["tx_kbps_trend"],
            "rx_kbps_trend": h["rx_kbps_trend"]
        }
        for h in hosts
    ] 

    # ---- last K decisions ----
    last_decisions = []
    try:
        with open(DECISION_LOG, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    last_decisions.append(json.loads(line))
        last_decisions = last_decisions[-last_k_decisions:]
    except FileNotFoundError:
        last_decisions = []

    # ---- HOST-level scene (what we show to LLM) ----
    scene = {
        "ts": now_iso(),
        "window_minutes": WINDOW_MINUTES,
        "trend_points": TREND_POINTS,
        "window_end_time": str(t_end),
        "counts": {
            "unique_host_macs": int(len(hosts)),
            "raw_rows_in_window": int(len(df)),
        },
        "host_metadata": host_meta_obj,
        "host_stats": hosts,
        "last_decisions": last_decisions
    }
    prompt = f"""
You are an SDN network anomaly detection function. You need the decide on the below task.

Task:
Select host MAC addresses that show increasing traffic.

Indicators of abnormal behavior:
- rx_pps_std > 0
- tx_pps_std unusually high
- tx_pps_delta large

If no host meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {{"mac":"MAC_ADDRESS","reason":"short_reason"}}
 ]
}}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{{' and end with '}}'.

You have to observe the host stats and outut the finding. You must follow the Rules and must respond in json file.
host_stats:
{json.dumps(hosts)}

Return JSON.
""".strip()

    # ---- log exactly what LLM saw ----
    with open(SCENE_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "ts": now_iso(),
            "model": model_name,
            "scene": scene,
            "prompt": prompt
        }) + "\n")

    print(prompt)
    return scene, prompt

In [ ]:
# ===================== CELL 2: call LLM (Ollama Cloud REST API) + store output =====================
import json
import requests
import time
import os

# ---- Cloud endpoint + auth ----
CLOUD_URL = "https://ollama.com/api/chat"
API_KEY = "23fbf0f676584a7983158ded37540f2c.9C4Aw8pI8jXQlC1vDCGIF7nb"

MODEL_NAME = "gpt-oss:20b-cloud"

scene, prompt = build_scene_and_prompt(
    model_name=MODEL_NAME,
    last_k_decisions=10
)

if prompt is None:
    print("[!] No data yet (empty/missing CSV rows).")
else:
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an SDN observer and you will detect data anomaly.\n"
                    "Return ONLY valid JSON. No prose."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        # optional knobs (cloud supports options like local)
        "options": {
            "temperature": 0,
            "top_p": 0.9
        }
    }

    headers = {
        "Authorization": f"Bearer {API_KEY.strip()}",
        "Content-Type": "application/json"
    }

    start_time = time.time()
    r = requests.post(CLOUD_URL, headers=headers, json=payload, timeout=360)

    print("Status:", r.status_code)
    if r.status_code != 200:
        print("Error body:", r.text[:500])

    r.raise_for_status()
    latency = time.time() - start_time

    response_json = r.json()

    # Ollama /api/chat returns: {"message": {"role": "...", "content": "..."} , ...}
    out = response_json.get("message", {}).get("content", "").strip()

    print("\n\n===== MODEL OUTPUT =====\n")
    print(out)
    print(f"\n[Latency: {latency:.2f} sec]")

    with open(DECISION_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "ts": now_iso(),
            "model": MODEL_NAME,
            "latency_sec": latency,
            "llm_raw": out,
            "scene_ts": scene.get("ts") if scene else None
        }) + "\n")

You are an SDN network anomaly detection function. You need the decide on the below task.

Task:
Select host MAC addresses that show increasing traffic.

Indicators of abnormal behavior:
- rx_pps_std > 0
- tx_pps_std unusually high
- tx_pps_delta large

If no host meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {"mac":"MAC_ADDRESS","reason":"short_reason"}
 ]
}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{' and end with '}'.

You have to observe the host stats and outut the finding. You must follow the Rules and must respond in json file.
host_stats:
[{"mac": "00:00:00:00:00:01", "tx_pps_trend": [0.67, 0.67, 0.6, 0.66, 0.67, 0.6, 0.67, 0.67, 0.6, 0.67], "rx_pps_trend": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "tx_kbps_trend": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,

## Better Version

In [6]:
import json, requests
from datetime import datetime
import pandas as pd
import numpy as np
from datetime import timedelta
import requests
import time
from langchain_ollama import ChatOllama
import json, time
import json
import requests
import time
import os

In [7]:
# ===================== CELL 1: build HOST-focused scene + store what will be shown to LLM =====================
HOST_CSV = "host_stats_onos.csv"          # <-- your host stats CSV
HOST_META = "host_metadata.json"    # <-- optional (mac->host mapping)
WINDOW_MINUTES = 5
TREND_POINTS = 10
SCENE_LOG = "llm_scene_log.jsonl"
DECISION_LOG = "llm_decision_log.jsonl"

def now_iso(): 
    return datetime.now().isoformat()

def build_host_stats(df, trend_points=10):
    """
    Input: df already filtered to the desired time window
    Output: list of host objects (one per MAC)
    """
    hosts = []
    for mac, g in df.groupby("host_mac"):
        g = g.sort_values("timestamp").tail(trend_points)

        tx_pps = g["tx_pps"].astype(float).values
        rx_pps = g["rx_pps"].astype(float).values

        tx_kbps = g["tx_mbps"].astype(float).values * 1000.0
        rx_kbps = g["rx_mbps"].astype(float).values * 1000.0

        if len(tx_pps) < 2:
            continue

        host_obj = {
            "mac": str(mac),

            "tx_pps_trend": [round(x, 2) for x in tx_pps.tolist()],
            "rx_pps_trend": [round(x, 2) for x in rx_pps.tolist()],

            "tx_kbps_trend": [round(x, 2) for x in tx_kbps.tolist()],
            "rx_kbps_trend": [round(x, 2) for x in rx_kbps.tolist()],

            "tx_pps_mean": round(float(np.mean(tx_pps)), 2),
            "rx_pps_mean": round(float(np.mean(rx_pps)), 2),

            "tx_pps_std": round(float(np.std(tx_pps)), 3),
            "rx_pps_std": round(float(np.std(rx_pps)), 3),

            "tx_pps_max": round(float(np.max(tx_pps)), 2),
            "rx_pps_max": round(float(np.max(rx_pps)), 2),

            "tx_kbps_mean": round(float(np.mean(tx_kbps)), 2),
            "rx_kbps_mean": round(float(np.mean(rx_kbps)), 2),

            "tx_kbps_std": round(float(np.std(tx_kbps)), 3),
            "rx_kbps_std": round(float(np.std(rx_kbps)), 3),

            "tx_kbps_max": round(float(np.max(tx_kbps)), 2),
            "rx_kbps_max": round(float(np.max(rx_kbps)), 2),

            "tx_pps_delta": round(float(tx_pps[-1] - tx_pps[-2]), 2),
            "rx_pps_delta": round(float(rx_pps[-1] - rx_pps[-2]), 2),

            "tx_kbps_delta": round(float(tx_kbps[-1] - tx_kbps[-2]), 2),
            "rx_kbps_delta": round(float(rx_kbps[-1] - rx_kbps[-2]), 2),
        }

        hosts.append(host_obj)

    return hosts

def build_host_scene_and_prompt(model_name="llama3", last_k_decisions=10):
    # ---- read host stats ----
    df = pd.read_csv(HOST_CSV)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    if df.empty:
        return None, None

    # ---- filter last WINDOW_MINUTES ----
    t_end = df["timestamp"].max()
    t_min = t_end - pd.Timedelta(minutes=WINDOW_MINUTES)
    dfw = df[df["timestamp"] >= t_min].copy()
    if dfw.empty:
        return None, None
    
    # ---- metadata ----
    try:
        with open(HOST_META, "r", encoding="utf-8") as f:
            host_meta_obj = json.load(f)
    except FileNotFoundError:
        host_meta_obj = {"note": f"{HOST_META} not found"}

    # ---- build host stats via separate function ----
    hosts = build_host_stats(dfw, trend_points=TREND_POINTS)

    #///// filter //////
    hosts = [
        {
            "mac": h["mac"],
            "tx_pps_trend": h["tx_pps_trend"],
            "rx_pps_trend": h["rx_pps_trend"],
            "tx_kbps_trend": h["tx_kbps_trend"],
            "rx_kbps_trend": h["rx_kbps_trend"]
        }
        for h in hosts
    ] 

    # ---- last K decisions ----
    last_decisions = []
    try:
        with open(DECISION_LOG, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    last_decisions.append(json.loads(line))
        last_decisions = last_decisions[-last_k_decisions:]
    except FileNotFoundError:
        last_decisions = []

    # ---- HOST-level scene (what we show to LLM) ----
    scene = {
        "ts": now_iso(),
        "window_minutes": WINDOW_MINUTES,
        "trend_points": TREND_POINTS,
        "window_end_time": str(t_end),
        "counts": {
            "unique_host_macs": int(len(hosts)),
            "raw_rows_in_window": int(len(df)),
        },
        "host_metadata": host_meta_obj,
        "host_stats": hosts,
        "last_decisions": last_decisions
    }
    prompt = f"""
You are an SDN network anomaly detection function. You need the decide on the below task.

Task:
Select host MAC addresses that show increasing traffic.

Indicators of abnormal behavior:
- rx_pps_std > 0
- tx_pps_std unusually high
- tx_pps_delta large

If no host meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {{"mac":"MAC_ADDRESS","reason":"short_reason"}}
 ]
}}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{{' and end with '}}'.

You have to observe the host stats and outut the finding. You must follow the Rules and must respond in json file.
host_stats:
{json.dumps(hosts)}

Return JSON.
""".strip()

    # ---- log exactly what LLM saw ----
    with open(SCENE_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "ts": now_iso(),
            "model": model_name,
            "scene": scene,
            "prompt": prompt
        }) + "\n")

    print(prompt)
    return scene, prompt

In [8]:
import json, time
from datetime import datetime
import numpy as np
import pandas as pd

LINK_CSV   = "link_stats_onos.csv"
WINDOW_MINUTES = 5
TREND_POINTS   = 10

SCENE_LOG    = "llm_scene_log_link.jsonl"
DECISION_LOG = "llm_decision_log_link.jsonl"

def now_iso():
    return datetime.now().isoformat()

def build_link_stats(df, trend_points=10):
    """
    Input: df already filtered to the desired time window
    Output: list of link objects (one per link_id)
    """
    links = []

    for link_id, g in df.groupby("link_id"):
        g = g.sort_values("timestamp").tail(trend_points)

        rx_pps = g["rx_pps"].astype(float).values
        tx_pps = g["tx_pps"].astype(float).values

        rx_kbps = g["rx_mbps"].astype(float).values * 1000.0
        tx_kbps = g["tx_mbps"].astype(float).values * 1000.0

        if len(rx_pps) < 2:
            continue

        link_obj = {
            "link_id": str(link_id),

            "rx_pps_trend": [round(x, 2) for x in rx_pps.tolist()],
            "tx_pps_trend": [round(x, 2) for x in tx_pps.tolist()],

            "rx_kbps_trend": [round(x, 2) for x in rx_kbps.tolist()],
            "tx_kbps_trend": [round(x, 2) for x in tx_kbps.tolist()],

            "rx_pps_mean": round(float(np.mean(rx_pps)), 2),
            "tx_pps_mean": round(float(np.mean(tx_pps)), 2),

            "rx_pps_std": round(float(np.std(rx_pps)), 3),
            "tx_pps_std": round(float(np.std(tx_pps)), 3),

            "rx_pps_max": round(float(np.max(rx_pps)), 2),
            "tx_pps_max": round(float(np.max(tx_pps)), 2),

            "rx_kbps_mean": round(float(np.mean(rx_kbps)), 2),
            "tx_kbps_mean": round(float(np.mean(tx_kbps)), 2),

            "rx_kbps_std": round(float(np.std(rx_kbps)), 3),
            "tx_kbps_std": round(float(np.std(tx_kbps)), 3),

            "rx_kbps_max": round(float(np.max(rx_kbps)), 2),
            "tx_kbps_max": round(float(np.max(tx_kbps)), 2),

            "rx_pps_delta": round(float(rx_pps[-1] - rx_pps[-2]), 2),
            "tx_pps_delta": round(float(tx_pps[-1] - tx_pps[-2]), 2),

            "rx_kbps_delta": round(float(rx_kbps[-1] - rx_kbps[-2]), 2),
            "tx_kbps_delta": round(float(tx_kbps[-1] - tx_kbps[-2]), 2),
        }

        links.append(link_obj)

    return links

def build_link_scene_and_prompt(model_name="llama3", last_k_decisions=10):
    df = pd.read_csv(LINK_CSV)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    if df.empty:
        return None, None

    # --- filter last WINDOW_MINUTES ---
    t_end = df["timestamp"].max()
    t_min = t_end - pd.Timedelta(minutes=WINDOW_MINUTES)
    dfw = df[df["timestamp"] >= t_min].copy()

    if dfw.empty:
        return None, None

    # --- build link stats ---
    links = build_link_stats(dfw, trend_points=TREND_POINTS)

    # --- filter only link_id + trends (like hosts) ---
    links = [
        {
            "link_id": l["link_id"],
            "rx_pps_trend": l["rx_pps_trend"],
            "tx_pps_trend": l["tx_pps_trend"],
            "rx_kbps_trend": l["rx_kbps_trend"],
            "tx_kbps_trend": l["tx_kbps_trend"]
        }
        for l in links
    ]

    # --- last K decisions ---
    last_decisions = []
    try:
        with open(DECISION_LOG, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    last_decisions.append(json.loads(line))
        last_decisions = last_decisions[-last_k_decisions:]
    except FileNotFoundError:
        last_decisions = []

    scene = {
        "ts": now_iso(),
        "window_minutes": WINDOW_MINUTES,
        "trend_points": TREND_POINTS,
        "window_end_time": str(t_end),
        "counts": {
            "unique_links": int(len(links)),
            "raw_rows_in_window": int(len(dfw)),
        },
        "link_stats": links,
        "last_decisions": last_decisions
    }

    prompt = f"""
You are an SDN network anomaly detection function for LINKS.

Task:
Select link_id values that show abnormal congestion (rising load / spikes).

Indicators of abnormal behavior:
- rx_pps_trend rising / spiking
- tx_pps_trend rising / spiking
- rx_kbps_trend rising / spiking
- tx_kbps_trend rising / spiking

If no link meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{{
 "decision": "reroute | do_nothing",
 "links_to_avoid": ["LINK_ID"],
 "confidence": 0.0,
 "observation": [
   {{"link_id":"LINK_ID","reason":"short_reason"}}
 ]
}}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{{' and end with '}}'.

link_stats:
{json.dumps(links)}

Return JSON.
""".strip()

    with open(SCENE_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "ts": now_iso(),
            "model": model_name,
            "scene": scene,
            "prompt": prompt
        }) + "\n")

    print(prompt)
    return scene, prompt

In [9]:
import requests, time, json

CLOUD_URL = "https://ollama.com/api/chat"
API_KEY = "23fbf0f676584a7983158ded37540f2c.9C4Aw8pI8jXQlC1vDCGIF7nb"
MODEL_NAME = "gpt-oss:20b-cloud"

def call_cloud_llm(prompt):

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "Return ONLY valid JSON. No text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "top_p": 0.9
        }
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    start = time.time()

    r = requests.post(CLOUD_URL, headers=headers, json=payload, timeout=300)
    r.raise_for_status()

    latency = time.time() - start

    out = r.json().get("message", {}).get("content", "").strip()

    return out, latency

In [10]:
scene, prompt = build_host_scene_and_prompt()

out, latency = call_cloud_llm(prompt)

print(out)
print("Latency for Host:", latency)


# ////////links


scene2, prompt2 = build_link_scene_and_prompt()

out2, latency2 = call_cloud_llm(prompt2)

print(out2)
print("Latency for Links:", latency2)

You are an SDN network anomaly detection function. You need the decide on the below task.

Task:
Select host MAC addresses that show increasing traffic.

Indicators of abnormal behavior:
- rx_pps_std > 0
- tx_pps_std unusually high
- tx_pps_delta large

If no host meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {"mac":"MAC_ADDRESS","reason":"short_reason"}
 ]
}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{' and end with '}'.

You have to observe the host stats and outut the finding. You must follow the Rules and must respond in json file.
host_stats:
[{"mac": "00:00:00:00:00:01", "tx_pps_trend": [0.73, 0.64, 0.6, 0.65, 0.66, 0.6, 0.67, 0.64, 0.6, 40.66], "rx_pps_trend": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "tx_kbps_trend": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0

In [11]:
# ===================== COMPACT FINAL FUSION CELL =====================

import json, re, time
from datetime import datetime
import pandas as pd

def get_flagged_entities_with_trends(out1, out2):
    """
    Reads host/link LLM outputs, extracts flagged MACs and link_ids,
    then fetches last WINDOW_MINUTES of trend data only for those entities.

    Returns:
        host_report
        link_report
        flagged_macs
        flagged_links
        flagged_host_stats
        flagged_link_stats
    """

    # ---------- parse LLM outputs ----------
    host_report = json.loads(out1) if isinstance(out1, str) else out1
    link_report = json.loads(out2) if isinstance(out2, str) else out2

    flagged_macs = host_report.get("macs_to_shuffle", []) if host_report.get("decision") == "ip_shuffle" else []
    flagged_links = link_report.get("links_to_avoid", []) if link_report.get("decision") == "reroute" else []

    flagged_macs_set = set(flagged_macs)
    flagged_links_set = set(flagged_links)

    # ---------- host side ----------
    flagged_host_stats = []
    hdf = pd.read_csv(HOST_CSV)
    hdf["timestamp"] = pd.to_datetime(hdf["timestamp"], errors="coerce")
    hdf = hdf.dropna(subset=["timestamp"])

    if not hdf.empty and flagged_macs_set:
        h_end = hdf["timestamp"].max()
        h_min = h_end - pd.Timedelta(minutes=WINDOW_MINUTES)
        hdfw = hdf[hdf["timestamp"] >= h_min].copy()

        if not hdfw.empty:
            all_hosts = build_host_stats(hdfw, trend_points=TREND_POINTS)
            flagged_host_stats = [
                {
                    "mac": h["mac"],
                    "tx_pps_trend": h["tx_pps_trend"],
                    "rx_pps_trend": h["rx_pps_trend"],
                    "tx_kbps_trend": h["tx_kbps_trend"],
                    "rx_kbps_trend": h["rx_kbps_trend"]
                }
                for h in all_hosts if h["mac"] in flagged_macs_set
            ]

    # ---------- link side ----------
    flagged_link_stats = []
    ldf = pd.read_csv(LINK_CSV)
    ldf["timestamp"] = pd.to_datetime(ldf["timestamp"], errors="coerce")
    ldf = ldf.dropna(subset=["timestamp"])

    if not ldf.empty and flagged_links_set:
        l_end = ldf["timestamp"].max()
        l_min = l_end - pd.Timedelta(minutes=WINDOW_MINUTES)
        ldfw = ldf[ldf["timestamp"] >= l_min].copy()

        if not ldfw.empty:
            all_links = build_link_stats(ldfw, trend_points=TREND_POINTS)
            flagged_link_stats = [
                {
                    "link_id": l["link_id"],
                    "rx_pps_trend": l["rx_pps_trend"],
                    "tx_pps_trend": l["tx_pps_trend"],
                    "rx_kbps_trend": l["rx_kbps_trend"],
                    "tx_kbps_trend": l["tx_kbps_trend"]
                }
                for l in all_links if l["link_id"] in flagged_links_set
            ]

    return (
        host_report,
        link_report,
        flagged_macs,
        flagged_links,
        flagged_host_stats,
        flagged_link_stats
    )

host_report, link_report, flagged_macs, flagged_links, flagged_host_stats, flagged_link_stats = \
    get_flagged_entities_with_trends(out, out2)

scene = {
    "host_report": host_report,
    "link_report": link_report,
    "flagged_host_stats": flagged_host_stats,
    "flagged_link_stats": flagged_link_stats,
    "policy_order": ["do_nothing", "rrm", "ip", "both"]
}

prompt = f"""
You are the final SDN mitigation decision function.

You are given:
1. a host anomaly report,
2. a link anomaly report,
3. the recent 5-minute trend data only for the hosts and links that were flagged.
4. select atleast two host and one link

Your job:
- Recheck the flagged host and link trends.
- Decide which flagged hosts and links remain true final candidates.
- Decide the final mitigation action.

Defense order:
do_nothing < rrm < ip < both

Meaning:
- do_nothing = lowest cost, weakest response
- rrm = stronger than do_nothing, lower cost than ip
- ip = stronger than rrm, higher cost than rrm
- both = strongest response, highest cost

Policy:
- Later actions are stronger defenses.
- Later actions also cost more.
- Choose the lowest-cost action that is still sufficient for the verified severity.
- Do not escalate to a stronger action unless the trends clearly justify it.
- Use both only when host-side abnormality and link-side abnormality are both clearly strong after recheck.
- If the reported abnormality is weak or unsupported after recheck, choose do_nothing.

Guidance:
- Prefer rrm when the main problem is link congestion or suspicious link rise.
- Prefer ip when the main problem is host-side suspicious traffic growth and host evidence is stronger than link-only evidence.
- Prefer both only when flagged hosts and flagged links both remain strongly abnormal and combined defense is justified.
- Return only final candidates that are still supported after recheck.
- If no candidate remains supported, return empty target lists and choose do_nothing.

Return ONLY valid JSON.

Output schema:
{{
  "final_decision": "do_nothing | rrm | ip | both",
  "final_macs": ["MAC_ADDRESS"],
  "final_links": ["LINK_ID"],
  "confidence": 0.0,
  "severity": "low | medium | high | critical",
  "observation": [
    {{"type":"host","id":"MAC_ADDRESS","reason":"short_reason"}},
    {{"type":"link","id":"LINK_ID","reason":"short_reason"}}
  ]
}}

Rules:
- Output JSON only.
- No explanation outside JSON.
- JSON must start with '{{' and end with '}}'.
- Keep only rechecked and supported final candidates.
- If action is do_nothing, return empty lists for final_macs and final_links.

Input:
{json.dumps(scene, ensure_ascii=False)}

Return JSON only.
""".strip()

final_out, final_latency = call_cloud_llm(prompt)
final_obj = json.loads(final_out)

print("FINAL RAW:", final_out)
print("FINAL JSON:", json.dumps(final_obj, indent=2))
print("FINAL LATENCY:", final_latency)


FINAL RAW: {
  "final_decision": "both",
  "final_macs": [
    "00:00:00:00:00:01",
    "00:00:00:00:00:02"
  ],
  "final_links": [
    "of:0000000000000002:1 -> of:000000000000000b:4",
    "of:0000000000000009:1 -> of:000000000000000c:6"
  ],
  "confidence": 0.93,
  "severity": "critical",
  "observation": [
    {
      "type": "host",
      "id": "00:00:00:00:00:01",
      "reason": "tx spike"
    },
    {
      "type": "host",
      "id": "00:00:00:00:00:02",
      "reason": "rx spike"
    },
    {
      "type": "link",
      "id": "of:0000000000000002:1 -> of:000000000000000b:4",
      "reason": "rx_kbps spike"
    },
    {
      "type": "link",
      "id": "of:0000000000000009:1 -> of:000000000000000c:6",
      "reason": "rx_kbps spike"
    }
  ]
}
FINAL JSON: {
  "final_decision": "both",
  "final_macs": [
    "00:00:00:00:00:01",
    "00:00:00:00:00:02"
  ],
  "final_links": [
    "of:0000000000000002:1 -> of:000000000000000b:4",
    "of:0000000000000009:1 -> of:000000000000000c

In [12]:
import pandas as pd

def path_selector(final_obj, hoplist_csv):
    """
    Return safe candidate paths only between the selected hosts in final_obj,
    while avoiding blocked links in both directions, and keeping only one
    direction per host pair (e.g., keep h1->h2, drop h2->h1).

    hoplist.csv columns:
        host1, host2, option_number, hop_count, src_mac, dst_mac, path
    """

    def normalize_link(link_str):
        left, right = [x.strip() for x in str(link_str).split("->")]
        return tuple(sorted([left, right]))

    selected_macs = set(final_obj.get("final_macs", []))
    blocked_links = set(
        normalize_link(x.strip()) for x in final_obj.get("final_links", [])
    )

    df = pd.read_csv(
        hoplist_csv,
        header=None,
        names=["host1", "host2", "option_number", "hop_count", "src_mac", "dst_mac", "path"]
    )

    # keep only rows where BOTH endpoints are among the selected hosts
    pair_df = df[
        (df["src_mac"].isin(selected_macs)) & (df["dst_mac"].isin(selected_macs))
    ].copy()

    if pair_df.empty:
        return []

    def path_is_safe(path_str):
        links = [x.strip() for x in str(path_str).split(",")]
        norm_links = [normalize_link(link) for link in links]
        return all(link not in blocked_links for link in norm_links)

    pair_df["is_safe"] = pair_df["path"].apply(path_is_safe)
    safe_df = pair_df[pair_df["is_safe"]].copy()

    if safe_df.empty:
        return []

    # treat h1->h2 and h2->h1 as the same host pair
    safe_df["pair_key"] = safe_df.apply(
        lambda r: tuple(sorted([r["host1"], r["host2"]])),
        axis=1
    )

    # keep only one direction per pair; alphabetical order keeps h1->h2 over h2->h1
    safe_df = safe_df.sort_values(
        ["pair_key", "host1", "host2", "hop_count", "option_number"]
    )
    safe_df = safe_df.drop_duplicates(subset=["pair_key"], keep="first")

    candidates = []
    for _, row in safe_df.iterrows():
        candidates.append({
            "host1": row["host1"],
            "host2": row["host2"],
            "option_number": int(row["option_number"]),
            "hop_count": int(row["hop_count"]),
            "src_mac": row["src_mac"],
            "dst_mac": row["dst_mac"],
            "path": row["path"]
        })

    return candidates

In [13]:
def do_ip_shuffle(macs):
    print("[IP SHUFFLE] Target MACs:", macs)
    # your real IP shuffle code here


def do_rrm(links, candidate_paths):
    print("[RRM] Avoid links:", links)
    print("[RRM] Candidate safe paths:")
    for p in candidate_paths:
        print(p)
    # your real reroute / install-path code here


def dispatch_mitigation(final_obj, hoplist_csv):
    decision = final_obj.get("final_decision", "do_nothing")
    macs = final_obj.get("final_macs", [])
    links = final_obj.get("final_links", [])

    print("[DECISION]", decision)
    print("MACs:", macs)
    print("Links:", links)

    candidate_paths = []
    if decision in ["rrm", "both"]:
        candidate_paths = path_selector(final_obj, hoplist_csv)

    if decision == "do_nothing":
        print("[ACTION] No action taken")

    elif decision == "rrm":
        do_rrm(links, candidate_paths)

    elif decision == "ip":
        do_ip_shuffle(macs)

    elif decision == "both":
        do_ip_shuffle(macs)
        do_rrm(links, candidate_paths)

    else:
        print("[ERROR] Unknown decision:", decision)

In [14]:
final_obj = json.loads(final_out)
dispatch_mitigation(final_obj, "hop_list.csv")

[DECISION] both
MACs: ['00:00:00:00:00:01', '00:00:00:00:00:02']
Links: ['of:0000000000000002:1 -> of:000000000000000b:4', 'of:0000000000000009:1 -> of:000000000000000c:6']
[IP SHUFFLE] Target MACs: ['00:00:00:00:00:01', '00:00:00:00:00:02']
[RRM] Avoid links: ['of:0000000000000002:1 -> of:000000000000000b:4', 'of:0000000000000009:1 -> of:000000000000000c:6']
[RRM] Candidate safe paths:
{'host1': 'h1', 'host2': 'h2', 'option_number': 2, 'hop_count': 3, 'src_mac': '00:00:00:00:00:01', 'dst_mac': '00:00:00:00:00:02', 'path': 'of:0000000000000001:1 -> of:000000000000000b:1, of:000000000000000b:6 -> of:0000000000000008:1, of:0000000000000008:2 -> of:0000000000000002:2'}
